# 03 - Silver: Tipagem, derivacao e classificacao de negocio

## Objetivo

Transformar os dados da Silver Staging em uma camada analítica estruturada.

As operacoes realizadas nesta etapa sao:

- tipagem explicita de datas, numeros e textos;
- separacao de codigo e descricao de municipio;
- criacao de colunas derivadas de negocio;
- classificacao de CID por capitulo e grupo;
- classificacao de especie por tipo de beneficio;
- sinalizacao de CID informado ou nao informado.

A tabela gerada e:

`afastamento_inss.silver.beneficios_concedidos`

In [0]:
# ============================================================
# IMPORTACOES
# ============================================================

from pyspark.sql.functions import (
    col,
    count,
    when,
    trim,
    lit,
    current_timestamp,
    try_to_date,
    regexp_replace,
    substring,
    split,
    upper
)
from pyspark.sql.types import (
    IntegerType,
    DoubleType
)

In [0]:
# ============================================================
# PARAMETROS
# ============================================================

TABELA_ORIGEM  = "afastamento_inss.silver.stg_beneficios_concedidos"
TABELA_DESTINO = "afastamento_inss.silver.beneficios_concedidos"

print(f"Origem : {TABELA_ORIGEM}")
print(f"Destino: {TABELA_DESTINO}")

## 1. Leitura da Silver Staging

A Silver sempre le da Staging e nunca da Bronze diretamente.

Isso garante que os tratamentos de qualidade ja foram aplicados
antes de qualquer transformacao de negocio.

In [0]:
df_staging = spark.table(TABELA_ORIGEM)

print(f"Linhas : {df_staging.count()}")
print(f"Colunas: {len(df_staging.columns)}")

## 2. Tipagem de datas

As datas no arquivo original estao no formato DD/MM/YYYY como string.

Excecao: dt_dcb contem o valor 00/00/0000 para beneficios sem
data de cessacao. Esse valor nao pode ser convertido para DateType
e ja foi mapeado para null na Silver Staging.

As colunas convertidas sao:
- dt_nascimento
- dt_dcb
- dt_ddb
- dt_dib

In [0]:
df_silver = df_staging

# Converter datas do formato DD/MM/YYYY para DateType
COLUNAS_DATA = [
    "dt_nascimento",
    "dt_dcb",
    "dt_ddb",
    "dt_dib"
]

for c in COLUNAS_DATA:
    df_silver = df_silver.withColumn(
        c,
        try_to_date(col(c), "dd/MM/yyyy")
    )

print("Tipagem de datas aplicada.")
print(f"Colunas convertidas: {COLUNAS_DATA}")

# Verificar quantos nulls gerados por datas invalidas
for c in COLUNAS_DATA:
    nulos = df_silver.filter(col(c).isNull()).count()
    print(f"  {c:<25} nulls: {nulos:>8,}")

## 3. Tipagem de campos numericos

O campo qt_sm_rmi usa virgula como separador decimal.
Antes de converter para DoubleType e necessario substituir
a virgula por ponto.

O campo qt_anos_contribuicao e um inteiro.
O valor 0 e valido e representa zero anos de contribuicao.

In [0]:
# qt_sm_rmi: virgula -> ponto -> double
df_silver = df_silver.withColumn(
    "qt_sm_rmi",
    regexp_replace(col("qt_sm_rmi"), ",", ".").cast(DoubleType())
)

# qt_anos_contribuicao: string -> integer
df_silver = df_silver.withColumn(
    "qt_anos_contribuicao",
    col("qt_anos_contribuicao").cast(IntegerType())
)

print("Tipagem numerica aplicada.")

## 4. Separacao de codigo e nome do municipio

O campo mun_resid armazena codigo e nome do municipio
no mesmo valor, separados por hifen.

Exemplo: 21504-SP-Sao Paulo

A separacao gera duas colunas:
- mun_cod  : codigo IBGE do municipio
- mun_nome : nome do municipio com UF

In [0]:
df_silver = df_silver.withColumn(
    "mun_cod",
    when(
        col("mun_resid").isNotNull(),
        split(col("mun_resid"), "-").getItem(0)
    ).otherwise(None)
)

df_silver = df_silver.withColumn(
    "mun_nome",
    when(
        col("mun_resid").isNotNull(),
        trim(
            regexp_replace(col("mun_resid"), r"^\d+-", "")
        )
    ).otherwise(None)
)

print("Separacao de municipio aplicada.")
display(
    df_silver
    .select("mun_resid", "mun_cod", "mun_nome")
    .filter(col("mun_resid").isNotNull())
    .limit(10)
)

## 5. Classificacao de CID

O campo cid_cod contem o codigo CID-10 do diagnostico.

Sao derivadas duas colunas:

cid_capitulo:
- Letra inicial do codigo CID
- Identifica o capitulo da CID-10
- Exemplos: F (mental), M (osteomuscular), K (digestivo)

cid_grupo:
- Classificacao relevante para o projeto
- mental        : capitulo F (transtornos mentais)
- osteomuscular : capitulo M (sistema osteomuscular)
- cardiovascular : capitulo I (aparelho circulatorio)
- respiratorio  : capitulo J (aparelho respiratorio)
- outros        : demais capitulos com CID informado
- nao_informado : CID ausente

O cid_capitulo e normalizado para maiuscula para corrigir
codigos CID registrados com letra minuscula na fonte.

cid_status:
- informado     : cid_cod preenchido
- nao_informado : cid_cod null

In [0]:
# cid_capitulo: primeira letra do cid_cod, normalizada para maiuscula
df_silver = df_silver.withColumn(
    "cid_capitulo",
    when(
        col("cid_cod").isNotNull(),
        upper(substring(col("cid_cod"), 1, 1))
    ).otherwise(None)
)

# cid_grupo: classificacao de negocio expandida
df_silver = df_silver.withColumn(
    "cid_grupo",
    when(col("cid_cod").isNull(),             "nao_informado")
    .when(col("cid_capitulo") == "F",         "mental")
    .when(col("cid_capitulo") == "M",         "osteomuscular")
    .when(col("cid_capitulo") == "I",         "cardiovascular")
    .when(col("cid_capitulo") == "J",         "respiratorio")
    .otherwise(                               "outros")
)

# cid_status
df_silver = df_silver.withColumn(
    "cid_status",
    when(col("cid_cod").isNotNull(), "informado")
    .otherwise("nao_informado")
)

print("Classificacao de CID aplicada.")
display(
    df_silver
    .groupBy("cid_grupo")
    .count()
    .orderBy("count", ascending=False)
)

## 6. Classificacao de especie por tipo de beneficio

O campo especie_cod identifica o tipo de beneficio concedido.

A coluna derivada tipo_beneficio agrupa as especies em categorias
relevantes para a analise de afastamentos:

- afastamento  : auxilio-doenca e auxilio-acidente
- aposentadoria: aposentadorias por idade, tempo, invalidez
- pensao       : pensao por morte
- assistencial : amparos sociais (BPC/LOAS)
- maternidade  : salario-maternidade
- outros       : demais especies

In [0]:
ESPECIES_AFASTAMENTO   = ["31", "91", "94", "71", "72"]
ESPECIES_APOSENTADORIA = ["41", "42", "46", "51", "52", "57"]
ESPECIES_PENSAO        = ["21", "56"]
ESPECIES_ASSISTENCIAL  = ["87", "88"]
ESPECIES_MATERNIDADE   = ["80"]

df_silver = df_silver.withColumn(
    "tipo_beneficio",
    when(col("especie_cod").isin(ESPECIES_AFASTAMENTO),   "afastamento")
    .when(col("especie_cod").isin(ESPECIES_APOSENTADORIA),"aposentadoria")
    .when(col("especie_cod").isin(ESPECIES_PENSAO),       "pensao")
    .when(col("especie_cod").isin(ESPECIES_ASSISTENCIAL), "assistencial")
    .when(col("especie_cod").isin(ESPECIES_MATERNIDADE),  "maternidade")
    .otherwise("outros")
)

print("Classificacao de especie aplicada.")
display(
    df_silver
    .groupBy("tipo_beneficio")
    .count()
    .orderBy("count", ascending=False)
)

## 7. Validacao pre-gravacao

Verificacao das colunas derivadas e tipos aplicados.

In [0]:
print("=" * 50)
print("SCHEMA DA SILVER")
print("=" * 50)
df_silver.printSchema()

print("=" * 50)
print("DISTRIBUICAO cid_grupo")
print("=" * 50)
display(
    df_silver
    .groupBy("cid_grupo", "cid_status")
    .count()
    .orderBy("count", ascending=False)
)

print("=" * 50)
print("DISTRIBUICAO tipo_beneficio")
print("=" * 50)
display(
    df_silver
    .groupBy("tipo_beneficio")
    .count()
    .orderBy("count", ascending=False)
)

## 8. Gravacao da Silver

In [0]:
(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA_DESTINO)
)

print(f"Tabela gravada: {TABELA_DESTINO}")

In [0]:
linhas_staging = df_staging.count()
linhas_silver  = spark.table(TABELA_DESTINO).count()
colunas_silver = len(spark.table(TABELA_DESTINO).columns)

print("=" * 50)
print("VALIDACAO STAGING vs SILVER")
print("=" * 50)
print(f"{'':30} {'STAGING':>8} {'SILVER':>8}")
print("-" * 50)
print(f"{'Linhas':30} {linhas_staging:>8,} {linhas_silver:>8,}")
print(f"{'Colunas Staging':30} {len(df_staging.columns):>8}")
print(f"{'Colunas Silver':30} {'':>8} {colunas_silver:>8}")
print("-" * 50)

if linhas_staging == linhas_silver:
    print("RESULTADO: OK - Nenhum registro perdido.")
else:
    diff = linhas_staging - linhas_silver
    print(f"RESULTADO: ATENCAO - Divergencia de {diff:,} registros.")

print("=" * 50)

display(spark.table(TABELA_DESTINO).limit(5))